# Figure 6a2 | Ordinary residual connections versus optimized DHC recipe

**Core conclusion examined:** a variant with lower held-out PPL should show an earlier rise in position-standardized top-5% functional correlation (FC).

**Within-group comparison:** The optimized recipe uses DHC×4, a token-adaptive final stream readout and training dropout 0.05 instead of 0.10. The readout is downstream of every FC tap, and dropout is disabled for every FC probe; neither directly alters the measured activation. Both variants use the same 24-layer, 176-wide backbone, tokens and FC estimator. This row tests the manuscript's performance–FC association, not isolated causal superiority of DHC.

The notebook is intentionally thin. Data handling, architectures, paired initialization, training, FC estimation, descriptive paired statistics and publication exports live in the shared `fc_compare` package so the four comparisons cannot silently drift apart.

> **Rerun note:** Rerunning uses the frozen a2-specific optimizer and fresh confirmation seeds 121/232/343/454/565. It writes to `results_v3/a2_dhc4_adaptive/`, preserving earlier diagnostics.

Training displays a per-seed progress bar with the latest paired losses.

## Preregistered measurement

- Corpus: the three local WikiText-2 splits; vocabulary is built from training only.
- PPL: complete validation/test splits, token-weighted cross-entropy, best validation checkpoint.
- FC units: the D-dimensional effective input to each selected block. `FC_LAYER = 'all'` computes FC independently in every layer and then takes their equal-weight mean; use a zero-based integer for one layer. For DHC, each tap is the actually routed branch input, not the mean of four streams.
- FC profile: fixed validation sequences; each feature is z-scored across sequences within each token position before concatenation.
- Primary FC statistic: mean of the largest signed 5% off-diagonal Pearson correlations.
- Earlier rise: first three-checkpoint sustained increase of at least 0.03 above step 0.
- Reporting: paired estimates and bootstrap confidence intervals are saved without automatic pass/fail decisions.
- Figure panels: absolute top-n% FC mean, its change from initialization, and validation PPL. All panels are drawn whenever training completes.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys

HERE = Path.cwd().resolve()
if not (HERE / 'fc_compare').is_dir():
    raise RuntimeError('Start Jupyter with FC_compare as the working directory.')
sys.path.insert(0, str(HERE))

from fc_compare import paper_config, run_comparison, smoke_config


In [ ]:
GROUP = 'a2'
DATA_DIR = Path('/mnt/Data16T/Data/haichao/code/AI_connectom/story/story_part2_struc_func/Transformer/data/wikitext-2')
MODE = 'paper'  # use 'smoke' only to test the pipeline; smoke results are not scientific evidence
FC_LAYER = 'all'  # 'all' averages per-layer FC; use a zero-based integer for one layer
OUTPUT_DIR = HERE / 'results_v3' / 'a2_dhc4_adaptive'

factory = paper_config if MODE == 'paper' else smoke_config
spec, data, train, fc = factory(GROUP, DATA_DIR)
fc = replace(fc, layer_selection=FC_LAYER)
print(spec)
print(data)
print(train)
print(fc)


In [ ]:
result = run_comparison(
    spec=spec,
    data=data,
    train=train,
    fc=fc,
    output_dir=OUTPUT_DIR,
)
print(json.dumps(result['aggregate'], indent=2, ensure_ascii=False))


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=result['figure_paths'][0]))


## Result inspection

The notebook always displays the three final panels after a successful run. Use `aggregate_summary.json` for the paired PPL change, FC-onset difference, early-FC AUC difference and their bootstrap confidence intervals; interpret these values together with the plotted trajectories.